# Logical Reasoning for Planning: Using an LLM to Construct and Test a Simple Planning Agent
Worked solution and report for the lab *Laboratory – Logical Reasoning for Planning* (warehouse robot, **Logic + Search = Planning**).

**How to use it:** run *Kernel → Restart & Run All*. The planner uses only the Python standard library. The optional Prolog tasks (6-8) call SWI-Prolog (`swipl`) if it is installed; if it is not, those cells print a message instead and you can paste the shown programs into SWI-Prolog or an online Prolog environment. All numbers quoted in the write-up come from the code cells; everything is deterministic.

| Section | Content |
|---|---|
| Task 0 | The planning problem, and which actions are applicable |
| Task 1 | A plan constructed by hand |
| Task 2 | LLM prompt and the generated BFS planner |
| Task 3 | Tests A, B, C (+ extra tests and seeded-bug tests) |
| Task 4 | Logic and search |
| Task 5 | Explanation vs. independent verification |
| Tasks 6-8 | Optional: Prolog as an independent verifier |
| - | Reflection questions and report notes |

**Conventions.** A state is a set of ground propositions such as `At(Robot,A)`. We use the *closed-world assumption*: a fact that is not in the state is false. Facts that an action does not mention are unchanged by it (the STRIPS assumption).

## Task 0: Understand the planning problem

**(a) Initial state:** $I=\{\text{At(Robot,A)},\ \text{At(Package,A)}\}$

**(b) Goal:** $G=\{\text{At(Package,C)}\}$. It is satisfied in any state $S$ with $S\models G$, i.e. $G\subseteq S$ (the robot's own location does not matter).

**(c, d) Actions, preconditions and effects.** The handout gives one example of each schema; we instantiate each schema for every location where it makes sense (the hand-made example in Task 1 mentions `PickUp(Package,B)`, so the planner must know that action exists and be able to reject it).

| Action | Preconditions | Negative effects (delete) | Positive effects (add) |
|---|---|---|---|
| `Move(x,y)` for $(x,y)\in\{(A,B),(B,A),(B,C),(C,B)\}$ | At(Robot,x) | At(Robot,x) | At(Robot,y) |
| `PickUp(Package,l)` for $l\in\{A,B,C\}$ | At(Robot,l), At(Package,l) | At(Package,l) | Holding(Package) |
| `Drop(Package,l)` for $l\in\{A,B,C\}$ | At(Robot,l), Holding(Package) | Holding(Package) | At(Package,l) |

That is 4 + 3 + 3 = 10 ground actions.

**Which actions are initially applicable?** An action is applicable in $S$ iff $S\models\mathit{Pre}(a)$; for sets of positive facts under the closed-world assumption that means $\mathit{Pre}(a)\subseteq S$.

- `PickUp(Package,A)`: preconditions {At(Robot,A), At(Package,A)} are both in $I$, so it **is applicable**.
- `Drop(Package,C)`: preconditions {At(Robot,C), Holding(Package)}; neither is in $I$ (the robot is at A, and the package is not held), so it is **not applicable**, even though it is in the list of available actions.

**Think about it.** Being listed as an available action says nothing about being executable *now*. Applicability is a logical question about the current state, and answering it is exactly where logical reasoning enters planning.

In [1]:
# Plain-set check of the Task 0 answers (closed-world: fact is true iff it is in the set)
I = {"At(Robot,A)", "At(Package,A)"}
pre_pickup_A = {"At(Robot,A)", "At(Package,A)"}
pre_drop_C = {"At(Robot,C)", "Holding(Package)"}

print("PickUp(Package,A) applicable in I?", pre_pickup_A <= I)
print("Drop(Package,C)   applicable in I?", pre_drop_C <= I, "| unsatisfied preconditions:", sorted(pre_drop_C - I))

PickUp(Package,A) applicable in I? True
Drop(Package,C)   applicable in I? False | unsatisfied preconditions: ['At(Robot,C)', 'Holding(Package)']


## Task 1: Construct a plan by hand

The package is at A, so it must be picked up *at A*; then the robot carries it A → B → C and drops it.

| State | Action leading to it | Facts |
|---|---|---|
| $S_0$ | (initial) | At(Robot,A), At(Package,A) |
| $S_1$ | PickUp(Package,A) | At(Robot,A), Holding(Package) |
| $S_2$ | Move(A,B) | At(Robot,B), Holding(Package) |
| $S_3$ | Move(B,C) | At(Robot,C), Holding(Package) |
| $S_4$ | Drop(Package,C) | At(Robot,C), At(Package,C) |

At every step the action's preconditions hold in the previous state, and $S_4\models G$, so this is a valid plan of length 4.

**A warning about the handout.** The handout suggests one might reason about `Move(A,B)`, `PickUp(Package,B)`, `Move(B,C)`, `Drop(Package,C)`. That sequence *looks* reasonable but is **not** a valid plan: after `Move(A,B)` the package is still at A, so the precondition At(Package,B) of `PickUp(Package,B)` is false. The validator in Test A below confirms this. It is a small example of "looks reasonable ≠ valid".

## Task 2: Ask an LLM to implement the planner

**Prompt used** (the handout's prompt, plus my own design requirements):

> I want to implement a simple planning agent in Python. Represent a state as a set of logical propositions (strings such as `"At(Robot,A)"`; use `frozenset` so states are hashable). Each action should contain: a name; positive preconditions; negative preconditions; positive effects; negative effects. An action is applicable if all of its preconditions are satisfied by the current state (all positive preconditions are in the state and no negative precondition is). When an action is applied: 1. remove its negative effects from the state; 2. add its positive effects to the state. Use breadth-first search to find a sequence of actions that achieves a specified goal (a set of propositions that must all hold). The program should also: detect when no plan exists; print the resulting sequence of actions; print the states reached after each action. Explain the implementation and identify any assumptions you make. Then run the program on this warehouse problem: locations A, B, C; the robot can move A↔B and B↔C; initial state {At(Robot,A), At(Package,A)}; goal {At(Package,C)}; actions Move, PickUp(Package, location), Drop(Package, location) with the preconditions and effects above.

*(Replace with your exact prompt if you used different wording.)*

**Where the specification appears in the program (the "Think About It" check):**

| Idea | Where in the code |
|---|---|
| Preconditions → when is an action applicable? | `applicable(state, a)`: `a.pos_pre <= state and not (a.neg_pre & state)` |
| Effects → how does the state change? | `apply_action(state, a)`: `(state - a.neg_eff) | a.pos_eff` (delete first, then add) |
| Goal → when does planning terminate? | `if goal <= s:` in `bfs_plan`, when a state is removed from the queue |
| BFS → how are alternative plans explored? | `deque` as a FIFO frontier, trying every applicable action in each state; the `parent` dictionary is the visited table and also stores the plan |

Below is the implementation after my inspection and edits (see the report notes at the end for what I changed).

In [2]:
from collections import deque
from dataclasses import dataclass

@dataclass(frozen=True)
class Action:
    name: str
    pos_pre: frozenset = frozenset()     # facts that must be in the state
    neg_pre: frozenset = frozenset()     # facts that must NOT be in the state
    pos_eff: frozenset = frozenset()     # add list
    neg_eff: frozenset = frozenset()     # delete list

def fs(*facts):
    return frozenset(facts)

# ---- the warehouse domain: ground actions ---------------------------------------------
LOCATIONS = ["A", "B", "C"]
CONNECTIONS = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]

def move(x, y):
    return Action(f"Move({x},{y})", pos_pre=fs(f"At(Robot,{x})"),
                  pos_eff=fs(f"At(Robot,{y})"), neg_eff=fs(f"At(Robot,{x})"))

def pickup(l):
    return Action(f"PickUp(Package,{l})", pos_pre=fs(f"At(Robot,{l})", f"At(Package,{l})"),
                  pos_eff=fs("Holding(Package)"), neg_eff=fs(f"At(Package,{l})"))

def drop(l):
    return Action(f"Drop(Package,{l})", pos_pre=fs(f"At(Robot,{l})", "Holding(Package)"),
                  pos_eff=fs(f"At(Package,{l})"), neg_eff=fs("Holding(Package)"))

ACTIONS = [move(x, y) for x, y in CONNECTIONS] + [pickup(l) for l in LOCATIONS] + [drop(l) for l in LOCATIONS]
INITIAL = fs("At(Robot,A)", "At(Package,A)")
GOAL = fs("At(Package,C)")

# ---- logic: applicability and effects ---------------------------------------------------
def applicable(state, a):
    """S |= Pre(a): positive preconditions all hold, negative preconditions all absent."""
    return a.pos_pre <= state and not (a.neg_pre & state)

def apply_action(state, a):
    """S' = (S - NegEffects) + PosEffects. Only legal when the action is applicable."""
    assert applicable(state, a), f"{a.name} is not applicable"
    return (state - a.neg_eff) | a.pos_eff

# ---- search: breadth-first -----------------------------------------------------------------
def bfs_plan(initial, goal, actions):
    initial, goal = frozenset(initial), frozenset(goal)
    frontier = deque([initial])
    parent = {initial: None}                      # state -> (previous state, action); doubles as visited set
    expanded = 0
    while frontier:
        s = frontier.popleft()
        if goal <= s:                             # goal test: S |= G
            plan, states = [], [s]
            while parent[s] is not None:
                prev, a = parent[s]
                plan.append(a.name); s = prev; states.append(s)
            return {"found": True, "plan": plan[::-1], "states": states[::-1],
                    "expanded": expanded, "generated": len(parent)}
        expanded += 1
        for a in actions:
            if applicable(s, a):
                s2 = apply_action(s, a)
                if s2 not in parent:
                    parent[s2] = (s, a)
                    frontier.append(s2)
    return {"found": False, "plan": None, "states": None, "expanded": expanded, "generated": len(parent)}

# ---- helpers: printing and an INDEPENDENT validator --------------------------------------
def fmt(state):
    return "{" + ", ".join(sorted(state)) + "}"

def validate_plan(initial, goal, plan, actions):
    """Replay `plan` from `initial` using plain Python sets; check every precondition explicitly.
    Written separately from applicable()/apply_action() so that it is an independent check."""
    table = {a.name: a for a in actions}
    facts, log = set(initial), []
    for i, name in enumerate(plan, 1):
        if name not in table:
            return False, log + [f"step {i}: {name} is not an available action"]
        a = table[name]
        missing = [p for p in sorted(a.pos_pre) if p not in facts]
        blocked = [p for p in sorted(a.neg_pre) if p in facts]
        if missing or blocked:
            return False, log + [f"step {i}: {name} NOT applicable (missing {missing}, forbidden-but-present {blocked})"]
        for f in a.neg_eff: facts.discard(f)
        for f in a.pos_eff: facts.add(f)
        log.append(f"step {i}: {name:18s} preconditions OK -> {fmt(facts)}")
    ok = set(goal) <= facts
    log.append("goal " + fmt(goal) + (" holds at the end" if ok else " does NOT hold at the end"))
    return ok, log

def show(title, initial, goal, actions):
    print(f"--- {title} ---")
    print("initial:", fmt(initial), "| goal:", fmt(goal))
    res = bfs_plan(initial, goal, actions)
    if not res["found"]:
        print("No plan found  (states expanded:", res["expanded"], ")")
        return res
    print("plan found:", res["plan"] if res["plan"] else "[] (goal already holds)", "| length", len(res["plan"]),
          "| states expanded:", res["expanded"])
    for i, st in enumerate(res["states"]):
        print(f"  S{i}: {fmt(st)}")
    ok, log = validate_plan(initial, goal, res["plan"], actions)
    print("independent validation:", "VALID" if ok else "INVALID")
    for line in log: print("   ", line)
    return res

print(f"{len(ACTIONS)} ground actions defined:", [a.name for a in ACTIONS])

10 ground actions defined: ['Move(A,B)', 'Move(B,A)', 'Move(B,C)', 'Move(C,B)', 'PickUp(Package,A)', 'PickUp(Package,B)', 'PickUp(Package,C)', 'Drop(Package,A)', 'Drop(Package,B)', 'Drop(Package,C)']


## Task 3: Test the generated planner

The LLM-generated program is not assumed correct. Expected results are written down *before* running each test.

### Test A: the original (solvable) warehouse problem
Expected: a 4-step plan equal to my hand-made plan from Task 1, with identical intermediate states.

In [3]:
resA = show("Test A: warehouse", INITIAL, GOAL, ACTIONS)

HAND_PLAN = ["PickUp(Package,A)", "Move(A,B)", "Move(B,C)", "Drop(Package,C)"]
HAND_STATES = [fs("At(Robot,A)", "At(Package,A)"), fs("At(Robot,A)", "Holding(Package)"),
               fs("At(Robot,B)", "Holding(Package)"), fs("At(Robot,C)", "Holding(Package)"),
               fs("At(Robot,C)", "At(Package,C)")]
print("\nmatches my hand-made plan:", resA["plan"] == HAND_PLAN)
print("matches my hand-computed states S0..S4:", resA["states"] == HAND_STATES)
assert resA["plan"] == HAND_PLAN and resA["states"] == HAND_STATES
expanded_A, generated_A = resA["expanded"], resA["generated"]

print("\n--- the handout's illustrative sequence (Move(A,B), PickUp(Package,B), ...) ---")
EXAMPLE_SEQ = ["Move(A,B)", "PickUp(Package,B)", "Move(B,C)", "Drop(Package,C)"]
ok, log = validate_plan(INITIAL, GOAL, EXAMPLE_SEQ, ACTIONS)
print("valid plan?", ok)
for line in log: print("   ", line)
assert not ok

--- Test A: warehouse ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Package,C)}
plan found: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)'] | length 4 | states expanded: 7
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Robot,A), Holding(Package)}
  S2: {At(Robot,B), Holding(Package)}
  S3: {At(Robot,C), Holding(Package)}
  S4: {At(Package,C), At(Robot,C)}
independent validation: VALID
    step 1: PickUp(Package,A)  preconditions OK -> {At(Robot,A), Holding(Package)}
    step 2: Move(A,B)          preconditions OK -> {At(Robot,B), Holding(Package)}
    step 3: Move(B,C)          preconditions OK -> {At(Robot,C), Holding(Package)}
    step 4: Drop(Package,C)    preconditions OK -> {At(Package,C), At(Robot,C)}
    goal {At(Package,C)} holds at the end

matches my hand-made plan: True
matches my hand-computed states S0..S4: True

--- the handout's illustrative sequence (Move(A,B), PickUp(Package,B), ...) ---
valid plan? False
    step 1: Move(A,B)          precondi

### Test B: impossible problem (the robot cannot pick up the package)
Expected: *No plan found*, and the program must terminate (not invent an action).

In [4]:
NO_PICKUP = [a for a in ACTIONS if not a.name.startswith("PickUp")]
resB = show("Test B: PickUp removed", INITIAL, GOAL, NO_PICKUP)
assert not resB["found"]

# a second impossible variant: the corridor to C is cut
NO_B_TO_C = [a for a in ACTIONS if a.name != "Move(B,C)"]
resB2 = show("Test B2: Move(B,C) removed", INITIAL, GOAL, NO_B_TO_C)
assert not resB2["found"]

--- Test B: PickUp removed ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Package,C)}
No plan found  (states expanded: 3 )
--- Test B2: Move(B,C) removed ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Package,C)}
No plan found  (states expanded: 6 )


### Test C: irrelevant actions (the robot can move without the package)
We add a shortcut `Move(A,C)` that moves only the robot, and check that "the robot reaches C" is not confused with "the package reaches C".

- **C1:** shortcut available but **no PickUp**, goal At(Package,C). Expected: no plan, even though the robot can reach C.
- **C2:** same actions, goal At(Robot,C). Expected: the one-step plan `Move(A,C)`, and the final state must **not** contain At(Package,C).
- **C3:** all actions plus the shortcut, goal At(Package,C). Expected: the shorter 3-step plan `PickUp, Move(A,C), Drop`.

In [5]:
SHORTCUT = move("A", "C")
resC1 = show("Test C1: shortcut, no PickUp, goal At(Package,C)", INITIAL, GOAL, NO_PICKUP + [SHORTCUT])
assert not resC1["found"]

print()
resC2 = show("Test C2: shortcut, no PickUp, goal At(Robot,C)", INITIAL, fs("At(Robot,C)"), NO_PICKUP + [SHORTCUT])
assert resC2["plan"] == ["Move(A,C)"] and "At(Package,C)" not in resC2["states"][-1]
print("package at C in the final state?", "At(Package,C)" in resC2["states"][-1], "(correctly False)")

print()
resC3 = show("Test C3: all actions + shortcut, goal At(Package,C)", INITIAL, GOAL, ACTIONS + [SHORTCUT])
assert resC3["plan"] == ["PickUp(Package,A)", "Move(A,C)", "Drop(Package,C)"]

--- Test C1: shortcut, no PickUp, goal At(Package,C) ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Package,C)}
No plan found  (states expanded: 3 )

--- Test C2: shortcut, no PickUp, goal At(Robot,C) ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Robot,C)}
plan found: ['Move(A,C)'] | length 1 | states expanded: 2
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Package,A), At(Robot,C)}
independent validation: VALID
    step 1: Move(A,C)          preconditions OK -> {At(Package,A), At(Robot,C)}
    goal {At(Robot,C)} holds at the end
package at C in the final state? False (correctly False)

--- Test C3: all actions + shortcut, goal At(Package,C) ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Package,C)}
plan found: ['PickUp(Package,A)', 'Move(A,C)', 'Drop(Package,C)'] | length 3 | states expanded: 7
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Robot,A), Holding(Package)}
  S2: {At(Robot,C), Holding(Package)}
  S3: {At(Package,C), At(Robot,C)}
independent validation: VA

### Extra tests (not required by the handout)

- **D, return trip:** goal {At(Package,C), At(Robot,A)}. The robot must deliver and come back, so the shortest plan has 6 steps. This checks that **delete effects** are really applied: the robot must stop being at A when it leaves.
- **E, trivial goal:** the goal already holds, so the empty plan `[]` is expected.
- **F, negative preconditions:** a toy `Switch` action that requires ¬On must be refused when `On` already holds.

In [6]:
resD = show("Test D: deliver and return", INITIAL, fs("At(Package,C)", "At(Robot,A)"), ACTIONS)
assert len(resD["plan"]) == 6
print()
resE = show("Test E: goal already true", INITIAL, fs("At(Package,A)"), ACTIONS)
assert resE["found"] and resE["plan"] == []
print()
SWITCH_ON = Action("Switch", neg_pre=fs("On"), pos_eff=fs("On"))
r1 = bfs_plan(fs(), fs("On"), [SWITCH_ON])
r2 = bfs_plan(fs("On"), fs("Done"), [SWITCH_ON])
print("Test F: from {} goal On ->", r1["plan"], "| from {On} the action is blocked, goal Done ->",
      "No plan found" if not r2["found"] else r2["plan"])
assert r1["plan"] == ["Switch"] and not r2["found"]

--- Test D: deliver and return ---
initial: {At(Package,A), At(Robot,A)} | goal: {At(Package,C), At(Robot,A)}
plan found: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)', 'Move(C,B)', 'Move(B,A)'] | length 6 | states expanded: 11
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Robot,A), Holding(Package)}
  S2: {At(Robot,B), Holding(Package)}
  S3: {At(Robot,C), Holding(Package)}
  S4: {At(Package,C), At(Robot,C)}
  S5: {At(Package,C), At(Robot,B)}
  S6: {At(Package,C), At(Robot,A)}
independent validation: VALID
    step 1: PickUp(Package,A)  preconditions OK -> {At(Robot,A), Holding(Package)}
    step 2: Move(A,B)          preconditions OK -> {At(Robot,B), Holding(Package)}
    step 3: Move(B,C)          preconditions OK -> {At(Robot,C), Holding(Package)}
    step 4: Drop(Package,C)    preconditions OK -> {At(Package,C), At(Robot,C)}
    step 5: Move(C,B)          preconditions OK -> {At(Package,C), At(Robot,B)}
    step 6: Move(B,A)          preconditions OK -> {At(Pack

### Seeded-bug tests: what does the test suite actually catch?
"Working output ≠ validated algorithm." We write three plausible faulty planners (the kind an LLM could produce) and run all tests on them, checking every returned plan with the independent validator:

1. **no precondition check:** every action is treated as applicable (the answer to reflection question 2);
2. **no delete effects:** only add effects are applied, so facts such as At(Robot,A) are never removed;
3. **no visited set:** BFS that never remembers states (with a cap on expansions so that an endless search is detected).

In [7]:
def buggy_plan(initial, goal, actions, bug, cap=5000):
    initial, goal = frozenset(initial), frozenset(goal)
    frontier = deque([(initial, [])])
    seen = {initial}
    expanded = 0
    while frontier:
        s, plan = frontier.popleft()
        if goal <= s:
            return {"found": True, "plan": plan}
        expanded += 1
        if expanded > cap:
            raise RuntimeError("search did not terminate")
        for a in actions:
            ok = True if bug == "no_precondition_check" else applicable(s, a)
            if not ok: continue
            s2 = (s | a.pos_eff) if bug == "no_delete_effects" else ((s - a.neg_eff) | a.pos_eff)
            if bug != "no_visited_check":
                if s2 in seen: continue
                seen.add(s2)
            frontier.append((s2, plan + [a.name]))
    return {"found": False, "plan": None}

SUITE = [  # name, initial, goal, actions, expected_found, expected_length
    ("A warehouse",      INITIAL, GOAL, ACTIONS, True, 4),
    ("B no PickUp",      INITIAL, GOAL, NO_PICKUP, False, None),
    ("B2 no Move(B,C)",  INITIAL, GOAL, NO_B_TO_C, False, None),
    ("C1 shortcut",      INITIAL, GOAL, NO_PICKUP + [SHORTCUT], False, None),
    ("C3 shortcut+all",  INITIAL, GOAL, ACTIONS + [SHORTCUT], True, 3),
    ("D return trip",    INITIAL, fs("At(Package,C)", "At(Robot,A)"), ACTIONS, True, 6),
    ("E trivial",        INITIAL, fs("At(Package,A)"), ACTIONS, True, 0),
]

def grade(solver):
    out = {}
    for name, init, goal, acts, exp_found, exp_len in SUITE:
        try:
            r = solver(init, goal, acts)
            ok = r["found"] == exp_found and (exp_len is None or len(r["plan"]) == exp_len)
            if r["found"]:
                ok = ok and validate_plan(init, goal, r["plan"], acts)[0]
        except RuntimeError:
            ok = False
        out[name] = ok
    return out

solvers = {"correct BFS": bfs_plan}
for b in ("no_precondition_check", "no_delete_effects", "no_visited_check"):
    solvers[b] = (lambda b: lambda i, g, a: buggy_plan(i, g, a, b))(b)

grades = {n: grade(f) for n, f in solvers.items()}
names = [s[0] for s in SUITE]
print(f"{'planner':24s}" + "".join(f"{n:>17s}" for n in names))
for n, g in grades.items():
    print(f"{n:24s}" + "".join(f"{('pass' if g[t] else 'FAIL'):>17s}" for t in names))

assert all(grades["correct BFS"].values())
for b in ("no_precondition_check", "no_delete_effects", "no_visited_check"):
    assert not all(grades[b].values()), b

# What do the faulty planners actually return?
print("\nno_precondition_check on Test A:", buggy_plan(INITIAL, GOAL, ACTIONS, "no_precondition_check")["plan"])
bad_A = buggy_plan(INITIAL, GOAL, ACTIONS, "no_delete_effects")["plan"]
print("no_delete_effects on Test A   :", bad_A, "-> validation:", validate_plan(INITIAL, GOAL, bad_A, ACTIONS)[0])
print("no_delete_effects on Test D   :", buggy_plan(INITIAL, fs("At(Package,C)", "At(Robot,A)"), ACTIONS, "no_delete_effects")["plan"])
bad = buggy_plan(INITIAL, fs("At(Package,C)", "At(Robot,A)"), ACTIONS, "no_delete_effects")["plan"]
print("   its validation:", validate_plan(INITIAL, fs("At(Package,C)", "At(Robot,A)"), bad, ACTIONS)[0],
      "(never actually returns the robot to A)" )

planner                       A warehouse      B no PickUp  B2 no Move(B,C)      C1 shortcut  C3 shortcut+all    D return trip        E trivial
correct BFS                          pass             pass             pass             pass             pass             pass             pass
no_precondition_check                FAIL             FAIL             FAIL             FAIL             FAIL             FAIL             pass
no_delete_effects                    FAIL             pass             pass             pass             pass             FAIL             pass
no_visited_check                     pass             FAIL             FAIL             FAIL             pass             pass             pass

no_precondition_check on Test A: ['Drop(Package,C)']
no_delete_effects on Test A   : ['Move(A,B)', 'Move(B,C)', 'PickUp(Package,A)', 'Drop(Package,C)'] -> validation: False
no_delete_effects on Test D   : ['Move(A,B)', 'Move(B,C)', 'PickUp(Package,A)', 'Drop(Package,C)']
   its 

**What the tests taught** (read the table above together with the printed plans):

- **No precondition check:** returns the one-step "plan" `Drop(Package,C)` for the original problem: the package "arrives" at C without the robot being there or holding anything. It is short and plausible but invalid, and it also "solves" the impossible problems, i.e. it invents actions. It passes only the trivial test E, where no action is needed.
- **No delete effects:** the planner believes the robot is still at A after it leaves, so it returns a 4-step plan that moves first and then does `PickUp(Package,A)` "from A". The plan has the right length but is invalid, and the independent validator rejects it; the return-trip test D fails for the same reason. A length check alone would not have noticed, which is why validation of the plan itself matters.
- **No visited set:** it passes every problem that has a solution (A, C3, D, E) but never terminates on the impossible problems (B, B2, C1). Only the "no plan exists" tests expose it.

So no single test catches all three faults: the solvable problem catches two, the impossible-problem tests catch the third, and independent validation of each returned plan is what turns a "plausible-looking" answer into a checked one.

## Task 4: Logic and search

**Completing the diagram:**

```
Current state
      ↓
Check action preconditions        (logic: S |= Pre(a) ?)
      ↓
Apply the action's effects        ← the "?" box:  S' = (S − NegEffects(a)) ∪ PosEffects(a)
      ↓
Generate successor state
      ↓
Search over alternatives          (BFS queue: which state to expand next)
      ↓
Goal?                             (logic again: S' |= G ?)
```

**In my own words.** The logical part answers *what is possible and what it changes*: given a state, which actions have all their preconditions satisfied, and what is the state after executing one? On its own it can only tell us about one step. The search part answers *what to try*: starting from the initial state it systematically explores the sequences of applicable actions (breadth-first, so shortest plans come first), remembers visited states so it does not loop, and stops when the logical test $S\models G$ succeeds. Without logic, search would not know which successors exist; without search, logic would know the legal moves but not which sequence leads to the goal. **Logic determines what is possible; search determines what to try.**

The relation to the previous module: the planner is the same BFS as in the search lab; a *state* is now a set of propositions, the *successor function* is "all applicable actions, with their effects applied", the *goal test* is entailment of the goal, and every step costs 1. (A heuristic could later turn this into A\* planning.)

The cell below makes the logical part visible: for each state on the plan it lists which of the 10 ground actions are applicable. It also counts the whole state space that the search can reach.

In [8]:
res = bfs_plan(INITIAL, GOAL, ACTIONS)
for i, st in enumerate(res["states"]):
    app = [a.name for a in ACTIONS if applicable(st, a)]
    print(f"S{i} = {fmt(st)}\n     applicable: {app}")

# Whole reachable state space (the thing BFS explores when the goal is unreachable)
def reachable(initial, actions):
    seen, q = {frozenset(initial)}, deque([frozenset(initial)])
    while q:
        s = q.popleft()
        for a in actions:
            if applicable(s, a):
                s2 = apply_action(s, a)
                if s2 not in seen: seen.add(s2); q.append(s2)
    return seen

n_states = len(reachable(INITIAL, ACTIONS))
print("\nreachable states in the warehouse:", n_states, "(robot in 3 places x package at 3 places or held = 12)")
print("states generated / expanded for Test A:", generated_A, "/", expanded_A)
assert n_states == 12

S0 = {At(Package,A), At(Robot,A)}
     applicable: ['Move(A,B)', 'PickUp(Package,A)']
S1 = {At(Robot,A), Holding(Package)}
     applicable: ['Move(A,B)', 'Drop(Package,A)']
S2 = {At(Robot,B), Holding(Package)}
     applicable: ['Move(B,A)', 'Move(B,C)', 'Drop(Package,B)']
S3 = {At(Robot,C), Holding(Package)}
     applicable: ['Move(C,B)', 'Drop(Package,C)']
S4 = {At(Package,C), At(Robot,C)}
     applicable: ['Move(C,B)', 'PickUp(Package,C)']

reachable states in the warehouse: 12 (robot in 3 places x package at 3 places or held = 12)
states generated / expanded for Test A: 10 / 7


**Think about it: "Logic determines what is possible; search determines what to try."** At $S_0$, ten actions exist but only `PickUp(Package,A)` and `Move(A,B)` are applicable (logic). BFS then keeps both alternatives in its queue and tries them level by level (search), until the goal test succeeds at depth 4. The reachable space has only 12 states, so BFS (which expanded 7 states for Test A) is more than enough here; in larger domains the number of states grows exponentially and better search or heuristics are needed.

## Task 5 (optional): Can the LLM verify its own plan?

**Prompt to the LLM:** *For every action in the plan, identify its preconditions and show that those preconditions are satisfied in the state in which the action is executed.*

I cannot reproduce *your* LLM's answer here, so the cell below uses an **illustrative explanation that I wrote to imitate a confident but flawed LLM answer** about the handout's tempting sequence `Move(A,B), PickUp(Package,B), Move(B,C), Drop(Package,C)`. Replace it with the real explanation from your LLM and compare it the same way: the explanation's claimed state after every step vs. the state computed by independently executing the actions.

In [9]:
SEQUENCE = ["Move(A,B)", "PickUp(Package,B)", "Move(B,C)", "Drop(Package,C)"]

# Claimed states, written the way a fluent explanation might narrate them ("the robot moves to B, picks the package up there...")
CLAIMED = [
    fs("At(Robot,B)", "At(Package,A)"),
    fs("At(Robot,B)", "Holding(Package)"),        # <- claims PickUp(Package,B) worked
    fs("At(Robot,C)", "Holding(Package)"),
    fs("At(Robot,C)", "At(Package,C)"),
]

table = {a.name: a for a in ACTIONS}
state, diverged = set(INITIAL), False
print(f"{'step':5s}{'action':20s}{'executable?':13s}explanation vs. execution")
for i, (name, claimed) in enumerate(zip(SEQUENCE, CLAIMED), 1):
    a = table[name]
    ok = applicable(frozenset(state), a)
    if ok:
        state = set(apply_action(frozenset(state), a))
        verdict = "states agree" if set(claimed) == state else f"MISMATCH: explanation says {fmt(claimed)}, execution gives {fmt(state)}"
    else:
        missing = sorted(a.pos_pre - frozenset(state))
        verdict = f"explanation claims success, but missing preconditions {missing} (state is {fmt(state)})"
        diverged = True
    print(f"{i:<5d}{name:20s}{str(ok):13s}{verdict}")
    if diverged: break

step action              executable?  explanation vs. execution
1    Move(A,B)           True         states agree
2    PickUp(Package,B)   False        explanation claims success, but missing preconditions ['At(Package,B)'] (state is {At(Package,A), At(Robot,B)})


**Which should you trust more: (a) the LLM's explanation, or (b) the independently executed state transitions?** **(b).** An LLM explanation is text generated to be plausible and coherent; it is not computed from the transition rules, so it can narrate a transition that never happens (here: picking up at B a package that is still at A). The execution applies exactly the stated preconditions and effects, deterministically and checkably, step by step. The explanation is still useful as a *hypothesis* and for readability, but only the independent check counts as verification: "a generated explanation is not the same as an independent verification". (The lesson is not that the LLM is useless, but that its output needs an independent check.)

## Optional extension: Prolog as a logical verifier

A Prolog program has *facts*, *rules* and *queries*. The cells below write each program to a file (`planner.pl`, as in the handout), run the queries with SWI-Prolog, and print the answers. The helper reports `true`/`false` for each query.

In [10]:
import shutil, subprocess, textwrap

SWIPL = shutil.which("swipl")
print("SWI-Prolog found:", SWIPL)

def run_prolog(program, queries, filename="planner.pl", show_program=True):
    program = textwrap.dedent(program).lstrip()
    with open(filename, "w") as f:
        f.write(program)
    if show_program:
        print(f"----- {filename} -----\n{program}-----------------------")
    if SWIPL is None:
        print("swipl not installed: paste the program above into SWI-Prolog (or an online Prolog) and ask the queries:")
        for q in queries: print("  ?-", q + ".")
        return None
    out = {}
    for q in queries:
        goal = f"({q} -> writeln(true) ; writeln(false))"
        p = subprocess.run([SWIPL, "-q", "-s", filename, "-g", goal, "-t", "halt"],
                           capture_output=True, text=True, timeout=60)
        lines = p.stdout.strip().splitlines()
        out[q] = lines[-1] if lines else ("error: " + p.stderr.strip())
        print(f"?- {q}.   =>  {out[q]}")
    return out

SWI-Prolog found: /usr/bin/swipl


### Task 6: Prolog as a plan verifier

In [11]:
warehouse_pl = """
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :-
    connected(X,Y).
"""
r6 = run_prolog(warehouse_pl, ["can_move(a,b)", "can_move(a,c)"])

----- planner.pl -----
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :-
    connected(X,Y).
-----------------------
?- can_move(a,b).   =>  true
?- can_move(a,c).   =>  false


**(a) Why does Prolog return `true` for `can_move(a,b)`?** The query matches the head of the rule `can_move(X,Y)` with X = a, Y = b, which leaves the subgoal `connected(a,b)`. That is a fact in the program, so the subgoal succeeds and, hence, so does the query.

**(b) Why does it not establish `can_move(a,c)`?** To prove it, Prolog would need `connected(a,c)`, but there is no such fact and no rule that could derive it. A Prolog answer of `false` means "cannot be proved from this program" (negation as failure under the closed-world assumption), not "is false in the real world". Note also that `c` is reachable from `a` in two moves; `can_move` describes only a *single* step, which is exactly the right notion for checking one action.

**(c) Relation between `can_move(X,Y)` and Connected(X,Y) → CanMove(X,Y).** The rule `can_move(X,Y) :- connected(X,Y).` is that implication, written right to left: the head is the conclusion, the body is the condition, and the variables are implicitly universally quantified, $\forall x\forall y\,(\mathit{Connected}(x,y)\rightarrow\mathit{CanMove}(x,y))$. A query asks whether the conclusion can be derived from the facts and rules (modus ponens).

### Task 7: Using Prolog to check a proposed plan

In [12]:
checker_pl = warehouse_pl + """
valid_move(X,Y) :-
    connected(X,Y).
"""
r7 = run_prolog(checker_pl, ["valid_move(a,b)", "valid_move(b,c)", "valid_move(a,c)"])
assert r7 is None or (r7["valid_move(a,b)"] == "true" and r7["valid_move(b,c)"] == "true" and r7["valid_move(a,c)"] == "false")

----- planner.pl -----
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :-
    connected(X,Y).

valid_move(X,Y) :-
    connected(X,Y).
-----------------------
?- valid_move(a,b).   =>  true
?- valid_move(b,c).   =>  true
?- valid_move(a,c).   =>  false


**Challenge: the Python planner proposes `Move(a,c)`.** The query `valid_move(a,c)` returns **false**, so Prolog does not support that action: the warehouse knowledge contains no `connected(a,c)`, so there is no direct A-to-C connection. (The shortcut `Move(A,C)` used in Test C is such an action: the Python planner accepts it only because we added it to its action list, whereas the Prolog description of the warehouse does not support it.) The Python program generated the candidate; Prolog independently checked it: **Generate → Independent verification**.

#### Going further: verify the *whole* plan in Prolog
Instead of checking individual moves, we can export the action definitions from Python as Prolog facts and let Prolog replay a complete plan, checking preconditions and applying effects. The Prolog checker below is written in a different language and style from the Python validator; we ask it about the planner's plan, about the handout's flawed sequence, and about the "no precondition check" bug's one-step plan.

In [13]:
def pl(x):                                     # "At(Robot,A)" -> "at(robot,a)"
    return x.lower()

def to_prolog_actions(actions):
    lst = lambda xs: "[" + ", ".join(sorted(pl(x) for x in xs)) + "]"
    return "\n".join(f"action({pl(a.name)}, {lst(a.pos_pre)}, {lst(a.neg_pre)}, {lst(a.pos_eff)}, {lst(a.neg_eff)})."
                     for a in actions)

PLAN_CHECKER_PL = r"""
% applicable(State, PosPre, NegPre): all positive preconditions hold, no negative one does
applicable(S, Pos, Neg) :- subset(Pos, S), \+ ( member(F, Neg), memberchk(F, S) ).
% apply_action(State, Add, Del, NewState): remove negative effects, then add positive ones
apply_action(S, Add, Del, S2) :- subtract(S, Del, S1), union(S1, Add, S2).
% valid_plan(State, Plan, Goal): replay the plan; every step must be applicable; the goal must hold at the end
valid_plan(S, [], Goal) :- subset(Goal, S).
valid_plan(S, [A|As], Goal) :-
    action(A, Pos, Neg, Add, Del),
    applicable(S, Pos, Neg),
    apply_action(S, Add, Del, S2),
    valid_plan(S2, As, Goal).
"""

init_pl = "[" + ", ".join(sorted(pl(f) for f in INITIAL)) + "]"
goal_pl = "[" + ", ".join(sorted(pl(f) for f in GOAL)) + "]"
as_plan = lambda names: "[" + ", ".join(pl(n) for n in names) + "]"

planner_plan = bfs_plan(INITIAL, GOAL, ACTIONS)["plan"]
queries = {
    "planner's plan":                  f"valid_plan({init_pl}, {as_plan(planner_plan)}, {goal_pl})",
    "handout's flawed sequence":       f"valid_plan({init_pl}, {as_plan(EXAMPLE_SEQ)}, {goal_pl})",
    "'no precondition check' plan":    f"valid_plan({init_pl}, {as_plan(['Drop(Package,C)'])}, {goal_pl})",
}
res_pl = run_prolog(to_prolog_actions(ACTIONS) + "\n" + PLAN_CHECKER_PL, list(queries.values()), filename="plan_checker.pl", show_program=False)
if res_pl is not None:
    print()
    for label, q in queries.items():
        print(f"{label:32s} -> Prolog says {res_pl[q]}")
    assert res_pl[queries["planner's plan"]] == "true"
    assert res_pl[queries["handout's flawed sequence"]] == "false"
    assert res_pl[queries["'no precondition check' plan"]] == "false"
print("\nfirst lines of the generated Prolog action facts:")
print("\n".join(to_prolog_actions(ACTIONS).splitlines()[:3]))

?- valid_plan([at(package,a), at(robot,a)], [pickup(package,a), move(a,b), move(b,c), drop(package,c)], [at(package,c)]).   =>  true
?- valid_plan([at(package,a), at(robot,a)], [move(a,b), pickup(package,b), move(b,c), drop(package,c)], [at(package,c)]).   =>  false
?- valid_plan([at(package,a), at(robot,a)], [drop(package,c)], [at(package,c)]).   =>  false

planner's plan                   -> Prolog says true
handout's flawed sequence        -> Prolog says false
'no precondition check' plan     -> Prolog says false

first lines of the generated Prolog action facts:
action(move(a,b), [at(robot,a)], [], [at(robot,b)], [at(robot,a)]).
action(move(b,a), [at(robot,b)], [], [at(robot,a)], [at(robot,b)]).
action(move(b,c), [at(robot,b)], [], [at(robot,c)], [at(robot,b)]).


The independent Prolog checker accepts the BFS planner's plan and rejects both the handout's tempting-but-invalid sequence and the one-step plan produced by the faulty planner. Three different implementations (the Python planner, the Python validator, the Prolog checker) agree on the correct plan and on the invalid ones, which is much stronger evidence than any one of them alone.

### Task 8: Connect Prolog to logical reasoning

In [14]:
road_pl = """
wet_road.
slippery :-
    wet_road.
reduce_speed :-
    slippery.
"""
r8 = run_prolog(road_pl, ["reduce_speed", "slippery", "wet_road"], filename="road.pl")
assert r8 is None or r8["reduce_speed"] == "true"

----- road.pl -----
wet_road.
slippery :-
    wet_road.
reduce_speed :-
    slippery.
-----------------------
?- reduce_speed.   =>  true
?- slippery.   =>  true
?- wet_road.   =>  true


**Why does `?- reduce_speed.` succeed?** To prove `reduce_speed`, Prolog uses the rule `reduce_speed :- slippery.`, which leaves the subgoal `slippery`. That subgoal uses the rule `slippery :- wet_road.`, leaving the subgoal `wet_road`, which is a fact. All subgoals are proved, so the query is proved.

**As a chain of implications (Fact ⇒ Rule ⇒ Rule ⇒ Conclusion):**

$$\mathit{WetRoad}\ \Rightarrow\ (\mathit{WetRoad}\rightarrow\mathit{Slippery})\ \Rightarrow\ (\mathit{Slippery}\rightarrow\mathit{ReduceSpeed})\ \Rightarrow\ \mathit{ReduceSpeed}$$

i.e. two applications of modus ponens: from WetRoad and WetRoad → Slippery we get Slippery; from Slippery and Slippery → ReduceSpeed we get ReduceSpeed. (Prolog actually searches *backwards* from the query, but the logical content is the same.)

**Think about it.** The programmer supplies facts and rules; the Prolog system performs the inference. But Prolog is not identical to classical logic: its execution involves unification and depth-first backtracking, and constructs such as negation as failure (`\+`) do not coincide with classical negation. That is why `false` above means "not provable from this program".

### Reflection (Prolog extension)

1. **Fact vs. rule.** A fact (`connected(a,b).`) states something that is unconditionally true. A rule (`can_move(X,Y) :- connected(X,Y).`) states that its head is true *if* its body can be proved; it is an implication and can produce new conclusions from other knowledge.
2. **Query = entailment question.** A query such as `?- reduce_speed.` asks whether the statement follows from the knowledge base (the facts and rules). `true` means a derivation was found; `false` means none exists in this knowledge base (not necessarily that the statement is false in the world).
3. **Why verify a Python plan in Prolog?** The Prolog check uses a separately written, declarative description of the world (connections, preconditions, effects). Errors in the planner's code, or in how it interprets the specification, are unlikely to be repeated identically in a second description, so disagreements reveal bugs (as in the check of the handout's sequence above).
4. **Advantage of an independent verifier for LLM-generated plans.** The generator (LLM-written code, or the LLM's own explanation) may be wrong while sounding convincing. A separate logical system that only needs the specification checks a candidate without trusting how it was produced, so a wrong plan is rejected rather than executed. Generation can be creative and fallible; checking can be simple and reliable.

## Reflection questions

**1. Why is it useful to specify action preconditions and effects before asking an LLM to write the planner?**
The preconditions and effects *are* the specification. With them written down, the LLM is asked to translate a precise design instead of inventing its own domain model, and I can check its code against something definite (e.g. the hand-made plan with its states). Without them, the LLM's guesses about what `PickUp` requires would silently become the definition of the problem.

**2. Give an example of an error that could occur if the planner failed to check an action's preconditions.**
It would accept `Drop(Package,C)` in the initial state and return a one-step "plan" in which the package arrives at C without the robot being at C or holding it (shown by the `no_precondition_check` planner in Task 3). It would also "solve" the impossible problem in which PickUp is removed.

**3. Why is a plan that "looks reasonable" not necessarily a valid plan?**
Plausibility is a property of how the plan reads; validity is a property of every step being applicable in the state where it is executed and the goal holding at the end. The handout's example `Move(A,B), PickUp(Package,B), ...` reads naturally but fails at step 2 because the package is still at A.

**4. What did the LLM contribute to the implementation?**
The first version of the code: the `Action` structure, the applicability and effect functions, the BFS loop with path reconstruction, and the printing of states, together with an explanation of its assumptions (closed world, STRIPS-style effects). That saved implementation time.

**5. What did you have to verify independently?**
That each action in the returned plan is applicable in the state where it is executed (independent validator, Prolog replay); that the plan and states equal my hand-made ones; that impossible problems give "No plan found" and terminate; that irrelevant actions do not make "robot at C" count as "package at C"; and that deletion effects really are applied (return-trip test).

**6. In this laboratory, where is logical reasoning being used?**
In the applicability test $S\models\mathit{Pre}(a)$, in the state update given by an action's effects, and in the goal test $S\models G$; and, in the extension, in the Prolog rules and queries used to verify moves and plans.

**7. How is planning related to the search algorithms studied in the previous module?**
Planning is search in the space of states: states are sets of propositions, the successor function is "apply each applicable action", the goal test is logical entailment of the goal, and each action costs 1. BFS here is the same algorithm as in the search lab; informed methods (A\* with a heuristic such as the number of unsatisfied goal facts) can be used for larger problems.

## Report notes: what was generated, modified, tested

*This is a draft based on the way the notebook was built. Edit it so that it describes your own session with the LLM and the prompts you actually used.*

| | Content |
|---|---|
| **I specified myself** | Task 0 (initial state, goal, actions with preconditions and effects, applicability answers), the hand-made plan with all states (Task 1), the expected results of every test |
| **LLM-generated** | The code skeleton: `Action`, `applicable`, `apply_action`, `bfs_plan`, printing of plan and states; the explanation of assumptions |
| **Modified by me** | Hashable `frozenset` states; delete effects applied before add effects; the `parent` table doubling as visited set (guarantees termination); goal test on removal from the queue; counting of expanded states; the separate validator; grounded action list for all locations |
| **Tested** | Tests A, B, C1-C3 from the handout; extra tests D-F; three seeded-bug planners; Prolog replay of the plan; the illustrative explanation check |

**Prompt appendix:** the planner prompt is given in Task 2. Additional prompts: *(Task 5)* "For every action in the plan, identify its preconditions and show that those preconditions are satisfied in the state in which the action is executed." *(optional)* "Write the Prolog facts and rules for the warehouse connections and a `valid_move/2` predicate."

**Takeaway:** Understand → Specify → Generate → Execute → Verify, and Logic + Search = Planning.